In [1]:
import os
import time

import pandas as pd

t0 = time.time()
from solution_utils import (
    compute_metrics, compute_partition_metrics, init_reaction_models, init_topic_models,
    predict_reaction, predict_topic, read_dialogs, read_labels, save_predictions,
)

In [ ]:
if os.environ.get("HW1_DATA") is None:
    os.environ["HW1_TRAIN"] = "data/hw1_train.jsonl"
    os.environ["HW1_DATA"] = "data/hw1_train.jsonl"
    os.environ["HW1_LABELS"] = "data/hw1_train.jsonl"
    os.environ["HW1_PREDICTIONS"] = "artifacts/predictions.csv"

In [3]:
TRAIN_PATH = os.environ.get("HW1_TRAIN", "hw1_train.jsonl")
DATA_PATH = os.environ.get("HW1_DATA", "hw1_train.jsonl")
LABELS_PATH = os.environ.get("HW1_LABELS", DATA_PATH)
PREDICTIONS_PATH = os.environ.get("HW1_PREDICTIONS", "predictions.csv")

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))

диалогов: 10000


## Задача 1. Реакция пользователя
Модели загружаются из `artifacts/` обучены в `eda.ipynb`. На скрытом тесте `HW1_LABELS` пустой, метрики не считаются.
При локальном запуске на train числа in-sample модели видели эти диалоги; честные оценки CV и отложенного теста лежат в `eda.ipynb` и README.

In [4]:
reaction_models = init_reaction_models(TRAIN_PATH)
reaction_labels = read_labels(LABELS_PATH, "reaction") if LABELS_PATH else {}

reaction_predictions, reaction_metrics = {}, {}
for name, model in reaction_models.items():
    reaction_predictions[name] = predict_reaction(model, dialogs)
    if reaction_labels:
        reaction_metrics[name] = compute_metrics(dialogs, reaction_predictions[name], reaction_labels)

pd.DataFrame(reaction_metrics).T.round(3)

,macro_f1,accuracy,balanced_accuracy
base_logreg,0.983,0.984,0.990
base_nb,0.803,0.837,0.761
base_svm,1.000,1.000,1.000
custom_turn_h8,0.895,0.915,0.939


## Задача 2. Темы
Модели тем загружаются из `artifacts/` обучены в `eda.ipynb`. Метрики разбиений (ARI, NMI, однородность, полнота, V-мера, чистота) считаются, только если переданы метки тем.

In [5]:
topic_models = init_topic_models(TRAIN_PATH)
topic_labels = read_labels(LABELS_PATH, "topic") if LABELS_PATH else {}

topic_predictions, topic_metrics = {}, {}
for name, model in topic_models.items():
    topic_predictions[name] = predict_topic(model, dialogs)
    if topic_labels:
        topic_metrics[name] = compute_partition_metrics(dialogs, topic_predictions[name], topic_labels)

pd.DataFrame(topic_metrics).T.round(3)

""


In [6]:
save_predictions(PREDICTIONS_PATH, dialogs, {"reaction": reaction_predictions, "topic": topic_predictions})
n_rows = len(dialogs) * (len(reaction_predictions) + len(topic_predictions))
print("строк в файле ответов:", n_rows)

строк в файле ответов: 80000


In [7]:
print("Всего: ", time.time() - t0,"сек")

Всего:  155.56314754486084 сек
